# The Theory: Cauchy–Schwarz, Hebbian Learning, and Zeroth-Order LoRA Post-Training

A self-contained, **runnable** theory notebook for the mathematics behind local, backprop-free (zeroth-order) post-training of low-rank (LoRA) adapters. It traces one thread — from an inequality, to an algorithm, to a set of quantitative laws that govern how expensive learning is — and **checks every key claim numerically** as it goes.

**Arc.**
1. The linear neuron and Hebbian plasticity (why plain Hebb is online PCA, and why it explodes).
2. The Cauchy–Schwarz inequality and its affine-function proof.
3. Cauchy–Schwarz *inside* a neuron: the bound `y² ≤ ‖w‖²‖x‖²` and its equality condition.
4. Efficiency and the Rayleigh quotient; the information ceiling `λ₁/Σλᵢ`.
5. Deriving Oja's rule from a constraint.
6. Real-analysis acceleration: Lagrange defect, Chebyshev/momentum, shift-invert, Rayleigh flatness, Robbins–Monro / Polyak–Ruppert.
7. The **bridge**: the alignment law `cos²θ = M/(M+P+1)` as a Cauchy–Schwarz cosine in parameter space.
8. The **node-space** alignment law `cos²θ = M/(M+d̄+1)`, `d̄ = (1−β)r + βm`.
9. The **Stage-B rank-sweep** prediction: estimation cost × approximation floor, the Eckart–Young knee, and a lower bound.
10. Variance-reduction levers, scientific positioning, and open proof obligations.

Every plot and table below is produced by the code cells — nothing is asserted without a check. Runs on CPU in a couple of minutes (`numpy` + `matplotlib` only).


## 0. Shared setup

We fix a symmetric input-covariance matrix `C` with a known eigenbasis so we can check every spectral claim against ground truth.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
n = 10
Q, _ = np.linalg.qr(rng.standard_normal((n, n)))          # random orthonormal eigenbasis
lam = np.array([4.0,2.0,1.0,0.8,0.6,0.5,0.4,0.3,0.2,0.1]) # eigenvalues (descending)
C = (Q * lam) @ Q.T                                        # C = Q diag(lam) Q^T
v1 = Q[:, 0]                                               # top eigenvector
Lc = np.linalg.cholesky(C + 1e-12*np.eye(n))
def sample(): return Lc @ rng.standard_normal(n)           # x ~ N(0, C)
print("eigenvalues:", lam)
print("lambda_1 / sum =", round(lam[0]/lam.sum(), 4), "(single-unit variance ceiling)")

## 1. Background: the linear neuron and Hebbian plasticity

A single artificial neuron turns inputs into one activation by **multiply, add, squish**:

$$s=\sum_i w_i x_i=\mathbf w^\top\mathbf x,\qquad z=s+b,\qquad y=\sigma(z)=\frac1{1+e^{-z}}\in(0,1).$$

A whole layer stacks each neuron's weights as a row of $\mathbf W$: $\mathbf z=\mathbf W\mathbf x+\mathbf b$, $\mathbf a=\sigma(\mathbf z)$.

**Hebb's rule** ("cells that fire together wire together") is the classic *unsupervised* signal. For pre-activity $x_i$ and post-activity $y$,

$$\Delta w_i=\eta\,x_i\,y,\qquad\text{i.e.}\qquad \Delta\mathbf w=\eta\,y\,\mathbf x=\eta\,(\mathbf w^\top\mathbf x)\,\mathbf x.$$

Here "linear" refers to the *update formula*, not to any assumption about the data. Averaged over inputs with covariance $\mathbf C=\mathbb E[\mathbf x\mathbf x^\top]$,

$$\mathbb E[\Delta\mathbf w]=\eta\,\mathbf C\,\mathbf w,$$

so $\mathbf w$ rotates toward the top eigenvector of $\mathbf C$: **plain Hebb is online PCA**. Its fatal flaw is that this dynamics has no finite-norm fixed point — $\|\mathbf w\|\to\infty$. The rest of the notebook is about *why*, *how* Cauchy–Schwarz both explains and fixes it, and *how far* real analysis lets us push the speed.

**Check.** Plain Hebb vs. Oja's rule (derived in §5): weight norm and alignment with $\mathbf v_1$.

In [ ]:
eta = 0.01; steps = 3000
w_h = 0.1*rng.standard_normal(n); w_o = w_h.copy()
nh, no, cos_o = [], [], []
for t in range(steps):
    x = sample()
    yh = w_h @ x; w_h = w_h + eta*yh*x                  # plain Hebb
    yo = w_o @ x; w_o = w_o + eta*yo*(x - yo*w_o)       # Oja
    nh.append(np.linalg.norm(w_h)); no.append(np.linalg.norm(w_o))
    cos_o.append(abs(w_o @ v1)/np.linalg.norm(w_o))
fig, ax = plt.subplots(1,2, figsize=(11,3.6))
ax[0].plot(nh, label="plain Hebb ||w||"); ax[0].plot(no, label="Oja ||w||")
ax[0].set_yscale("log"); ax[0].set_xlabel("step"); ax[0].set_title("weight norm"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(cos_o, color="C2"); ax[1].set_xlabel("step"); ax[1].set_ylabel("|cos(w, v1)|")
ax[1].set_title("Oja aligns with the top eigenvector"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Hebb ||w|| -> {nh[-1]:.1f} (blows up);  Oja ||w|| -> {no[-1]:.3f} (~1);  Oja |cos(w,v1)| -> {cos_o[-1]:.4f}")

## 2. The Cauchy–Schwarz inequality

**Theorem (Cauchy–Schwarz).** For vectors $\mathbf w,\mathbf x\in\mathbb R^n$ (or square-integrable functions),

$$|\mathbf w^\top\mathbf x|\le\|\mathbf w\|\,\|\mathbf x\|,\qquad \Big|\int_a^b fg\,dx\Big|\le\sqrt{\int_a^b f^2\,dx}\,\sqrt{\int_a^b g^2\,dx},$$

with **equality iff** $\mathbf w$ and $\mathbf x$ (resp. $f,g$) are *linearly dependent*.

**The affine-function proof.** Introduce a real parameter $t$ and an affine combination, then read off a discriminant. For functions,

$$F(t)=\int_a^b\big(tf(x)+g(x)\big)^2\,dx\ \ge\ 0\quad\text{for all }t,$$

because the integrand is a square. Expanding gives a quadratic $F(t)=At^2+2Bt+C$ with $A=\int f^2$, $B=\int fg$, $C=\int g^2$. A quadratic that is nonnegative for all real $t$ has at most one real root, so its discriminant is nonpositive:

$$\Delta=(2B)^2-4AC\le 0\ \Longrightarrow\ B^2\le AC,$$

which is exactly Cauchy–Schwarz. Equality forces $\Delta=0$, i.e. some $t_\ast$ with $t_\ast f+g\equiv 0$ almost everywhere: linear dependence. The identical argument with sums gives the vector form — the version that lives inside every linear neuron.

In [ ]:
w = rng.standard_normal(6); x = rng.standard_normal(6)
lhs, rhs = (w@x)**2, (w@w)*(x@x)
print(f"(w.x)^2 = {lhs:.3f}   <=   ||w||^2 ||x||^2 = {rhs:.3f}   ->  {lhs<=rhs}")
xdep = 2.7*w
print(f"equality when x = alpha*w:  (w.x)^2={ (w@xdep)**2:.3f}  vs  ||w||^2||x||^2={(w@w)*(xdep@xdep):.3f}")

## 3. Cauchy–Schwarz inside a neuron

Run the affine-function proof with the neuron's own vectors. For real $t$,

$$F(t)=\sum_i(t\,w_i+x_i)^2=\|\mathbf w\|^2t^2+2(\mathbf w^\top\mathbf x)\,t+\|\mathbf x\|^2\ \ge\ 0,$$

so the discriminant condition yields

$$\boxed{\,y^2=(\mathbf w^\top\mathbf x)^2\le\|\mathbf w\|^2\,\|\mathbf x\|^2,\qquad\text{equality}\iff\mathbf w=\alpha\,\mathbf x.\,}$$

That equality condition is the **geometric content of Hebb's rule**: a neuron is maximally responsive *exactly* when its weight vector aligns with the direction its inputs come from. Cauchy–Schwarz does two jobs at once: it **names the target** of Hebbian learning and **caps** the achievable response.

**Why plain Hebb explodes.** Tracking the squared norm under Hebb,

$$\Delta\|\mathbf w\|^2\approx 2\eta\,y\,(\mathbf w^\top\mathbf x)=2\eta\,(\mathbf w^\top\mathbf x)^2\ge 0,$$

so the magnitude strictly increases, and Cauchy–Schwarz caps the *growth rate*, $\Delta\|\mathbf w\|^2\le 2\eta\|\mathbf w\|^2\|\mathbf x\|^2$. Nothing pushes back — so the weights run away, exactly as the §1 plot showed. The fix is to add the term the equality condition $\|\mathbf w\|\to1$ points at.

## 4. Efficiency and the Rayleigh quotient

If "bigger response" cannot mean "bigger $\mathbf w^\top\mathbf C\mathbf w$" (that is unbounded), it should mean **response captured per unit weight cost**:

$$\rho(\mathbf w)=\frac{\mathbb E[y^2]}{\|\mathbf w\|^2}=\frac{\mathbf w^\top\mathbf C\,\mathbf w}{\mathbf w^\top\mathbf w},\qquad \mathbf C=\mathbb E[\mathbf x\mathbf x^\top].$$

For a single repeated input pattern, $\mathbf C=\mathbf x\mathbf x^\top$ is rank one and $\rho$ *is* Cauchy–Schwarz. For many inputs one needs the eigenvalue generalization (Rayleigh–Ritz). With $\mathbf C=\sum_i\lambda_i\mathbf v_i\mathbf v_i^\top$, $\lambda_1\ge\lambda_2\ge\cdots$, and $c_i=\mathbf v_i^\top\mathbf w$,

$$\rho(\mathbf w)=\frac{\sum_i\lambda_i c_i^2}{\sum_i c_i^2}\le\lambda_1,\qquad \rho_{\text{norm}}=\frac{\lambda_1}{\operatorname{tr}\mathbf C}=\frac{\lambda_1}{\sum_i\lambda_i}\le 1.$$

The maximum $\lambda_1$ is attained exactly at $\mathbf w\parallel\mathbf v_1$, and $\lambda_1/\sum_i\lambda_i$ is the honest efficiency number: the **fraction of total input variance a single linear unit can capture**.

In [ ]:
vals = [ (w@C@w)/(w@w) for w in rng.standard_normal((6,n)) ]
print("random Rayleigh quotients (all <= lambda_1 =", lam[0], "):")
print("   ", [round(v,3) for v in vals])
print("at v1:  rho =", round((v1@C@v1)/(v1@v1), 4), "= lambda_1")
print("normalized ceiling rho_norm = lambda_1/sum =", round(lam[0]/lam.sum(), 4))

## 5. Deriving Oja's rule from a constraint

The efficient version of Hebb should climb $\rho$: maximize $\mathbf w^\top\mathbf C\mathbf w$ subject to $\|\mathbf w\|=1$ (the normalization the Cauchy–Schwarz equality condition points at). The Lagrangian

$$L(\mathbf w,\lambda)=\mathbf w^\top\mathbf C\mathbf w-\lambda(\mathbf w^\top\mathbf w-1),\qquad \nabla_{\mathbf w}L=2\mathbf C\mathbf w-2\lambda\mathbf w$$

gives projected gradient ascent $\Delta\mathbf w\propto\mathbf C\mathbf w-(\mathbf w^\top\mathbf C\mathbf w)\mathbf w$. Replacing population quantities by instantaneous online estimates $\mathbf C\mathbf w\to\mathbf x y$ and $\mathbf w^\top\mathbf C\mathbf w\to y^2$ yields

$$\boxed{\ \Delta\mathbf w=\eta\,y\,(\mathbf x-y\,\mathbf w)\ }\qquad\text{(Oja's rule).}$$

This is not a different principle: it is Hebb's $\eta y\mathbf x$ minus exactly the normalization $\eta y^2\mathbf w$ the equality condition demands. It drives $\|\mathbf w\|\to1$, converges to $\mathbf v_1$, and attains the $\rho$ ceiling — bounded weights, same fixed point as Hebb (verified in §1).

**A more literal handle: normalize the signal.** Since $y=\mathbf w^\top\mathbf x\le\|\mathbf w\|\|\mathbf x\|$, the Cauchy–Schwarz ratio itself is a bounded, dimensionless learning signal

$$\hat y=\frac{\mathbf w^\top\mathbf x}{\|\mathbf w\|\,\|\mathbf x\|}=\cos\theta\in[-1,1],\qquad \Delta\mathbf w=\eta\,\hat y\,\frac{\mathbf x}{\|\mathbf x\|},$$

scale-invariant in both input and weight magnitude, so it cannot blow up and needs no per-input learning-rate tuning.

## 6. Real-analysis tools to accelerate or work around the bound

You cannot beat Cauchy–Schwarz: $y^2\le\|\mathbf w\|^2\|\mathbf x\|^2$ is a theorem and $\cos\theta>1$ is not on the menu. "Working around the bound" honestly means one of three things — make the slack explicit and attack it, change the geometry the bound is measured in, or recognize that the limit on *speed* is not Cauchy–Schwarz but the **spectral gap** — and go after that.

### 6.1 The defect is a sum of squares: Lagrange's identity

The gap in Cauchy–Schwarz is computable exactly. Lagrange's identity gives

$$\|\mathbf w\|^2\|\mathbf x\|^2-(\mathbf w^\top\mathbf x)^2=\sum_{i<j}(w_ix_j-w_jx_i)^2=\|\mathbf w\|^2\|\mathbf x\|^2\sin^2\theta=\det\begin{pmatrix}\|\mathbf w\|^2&\mathbf w^\top\mathbf x\\ \mathbf w^\top\mathbf x&\|\mathbf x\|^2\end{pmatrix}.$$

The inequality is an equality *plus* an explicit nonnegative remainder — the **Gram determinant**, the squared area of the parallelogram spanned by $\mathbf w$ and $\mathbf x$. "Closing the bound" is driving this remainder to zero, and Oja's rule is gradient descent on it under normalization. The sum-of-squares form even hands you the gradient in closed form.

### 6.2 The real bottleneck is $\lambda_2/\lambda_1$: Chebyshev / momentum

Plain Hebb/Oja is power iteration: after $k$ steps the off-target components decay like $(\lambda_2/\lambda_1)^k$. **That ratio, not the bound, is what makes it slow.** The real-analysis fix is the extremal (minimax / equioscillation) property of Chebyshev polynomials: among degree-$k$ polynomials pinned at $\lambda_1$, the Chebyshev polynomial stays smallest across the unwanted interval $[\lambda_n,\lambda_2]$, so applying $p_k(\mathbf C)$ instead of $\mathbf C^k$ suppresses that spectrum far faster.

You get $p_k(\mathbf C)$ for free from **momentum**. A heavy-ball power iteration $\mathbf w_{k+1}=\mathbf C\mathbf w_k-\beta\,\mathbf w_{k-1}$ (optimal $\beta\approx\lambda_2^2/4$ with $\lambda_1$ scaled to 1) produces iterates that are exactly degree-$k$ polynomials in $\mathbf C$, the Chebyshev filter. With relative gap $\Delta=(\lambda_1-\lambda_2)/\lambda_1$, this turns

$$O\!\Big(\tfrac1\Delta\log\tfrac1\epsilon\Big)\ \longrightarrow\ O\!\Big(\tfrac1{\sqrt\Delta}\log\tfrac1\epsilon\Big),$$

the classic square-root speedup — the single biggest practical lever.

**Check.** The cleanest, exact form of this speedup is gradient descent vs. heavy-ball (momentum) minimizing a quadratic $\tfrac12\mathbf x^\top H\mathbf x$ with condition number $\kappa$: GD needs $O(\kappa)$ iterations, momentum $O(\sqrt\kappa)$. Gradient descent here is itself a linear iteration $\mathbf x_{k+1}=(I-\alpha H)\mathbf x_k$ whose convergence is governed by the spectrum exactly as power iteration is — and power iteration for the top eigenvector (Oja) accelerates by the identical Chebyshev-polynomial principle. We expect log–log slopes $\approx+1$ and $\approx+\tfrac12$.

In [ ]:
def iters_quad(kappa, momentum, tol=1e-5, kmax=200000):
    nq = 20
    ev = np.linspace(1.0, kappa, nq)                      # eigenvalues in [1, kappa]
    Qk, _ = np.linalg.qr(rng.standard_normal((nq, nq))); Hk = (Qk*ev)@Qk.T
    mu, Lm = 1.0, float(kappa)
    if momentum:                                          # optimal heavy-ball
        alpha = 4/(np.sqrt(Lm)+np.sqrt(mu))**2
        beta  = ((np.sqrt(Lm)-np.sqrt(mu))/(np.sqrt(Lm)+np.sqrt(mu)))**2
    else:                                                 # optimal gradient descent
        alpha, beta = 2/(Lm+mu), 0.0
    x = rng.standard_normal(nq); xp = x.copy(); x0 = np.linalg.norm(x)
    for k in range(1, kmax+1):
        xn = x - alpha*(Hk@x) + beta*(x-xp); xp = x; x = xn
        if np.linalg.norm(x) < tol*x0: return k
    return kmax
kappas = np.array([10,30,100,300,1000])
gd = np.array([iters_quad(k, False) for k in kappas])
hb = np.array([iters_quad(k, True ) for k in kappas])
sgd = np.polyfit(np.log(kappas), np.log(gd), 1)[0]; shb = np.polyfit(np.log(kappas), np.log(hb), 1)[0]
plt.figure(figsize=(6,4))
plt.loglog(kappas, gd, "o-", label=f"gradient descent (slope {sgd:.2f})")
plt.loglog(kappas, hb, "s-", label=f"heavy-ball momentum (slope {shb:.2f})")
plt.xlabel("condition number kappa"); plt.ylabel("iterations to 1e-5"); plt.legend(); plt.grid(alpha=.3, which="both")
plt.title("Momentum: O(kappa) -> O(sqrt kappa)"); plt.tight_layout(); plt.show()
print(f"GD slope ~ {sgd:.2f} (predicted +1),  momentum slope ~ {shb:.2f} (predicted +0.5)")

### 6.3 The aggressive cousin: shift-invert / Rayleigh Quotient Iteration

A spectral transformation reshapes the gap directly. The map $\mathbf C\mapsto(\mathbf C-\sigma\mathbf I)^{-1}$ sends $\lambda_i\mapsto1/(\lambda_i-\sigma)$; placing $\sigma$ just below $\lambda_1$ blows the target eigenvalue up relative to everything else, so the effective gap becomes enormous. Feeding the shift adaptively from the Rayleigh quotient,

$$\mathbf w_{k+1}\propto(\mathbf C-\rho(\mathbf w_k)\mathbf I)^{-1}\mathbf w_k,$$

gives Rayleigh Quotient Iteration, which converges **cubically** near the solution. The price is a linear solve per step — the "expensive but ferocious" end, not a synaptic-local rule.

### 6.4 Efficiency saturates before the direction does: Rayleigh flatness

Because $\mathbf v_1$ is a critical point of $\rho$, its gradient vanishes there and

$$\rho(\mathbf v_1+\epsilon\,\mathbf u)=\lambda_1-\epsilon^2\sum_j(\lambda_1-\lambda_j)\langle\mathbf u,\mathbf v_j\rangle^2+O(\epsilon^3).$$

A first-order-accurate **direction** gives a second-order-accurate **efficiency**: you don't need $\mathbf w$ pinned to $\mathbf v_1$ to sit near peak $\rho$. Practically, stop early on direction and still bank near-optimal efficiency.

### 6.5 The stochastic online case: Robbins–Monro, Polyak–Ruppert

Two conditions govern the noisy update. The **Robbins–Monro** schedule needs enough drive but square-summable noise, $\sum_t\eta_t=\infty$, $\sum_t\eta_t^2<\infty$ (e.g. $\eta_t\propto1/t$). Then **Polyak–Ruppert averaging** — run a slightly larger constant step and report $\bar{\mathbf w}_T=\tfrac1T\sum_{t\le T}\mathbf w_t$ — delivers the variance reduction of a law of large numbers for free, hitting the optimal asymptotic rate without hand-tuning the schedule.

**Check (Rayleigh flatness).** $(\lambda_1-\rho(\mathbf v_1+\epsilon\mathbf u))/\epsilon^2$ should approach a constant as $\epsilon\to0$ (second-order flat).

In [ ]:
u = rng.standard_normal(n); u -= (u@v1)*v1; u /= np.linalg.norm(u)   # unit, orthogonal to v1
print(f"{'eps':>7} {'lam1 - rho':>12} {'ratio /eps^2':>14}")
for eps in [0.4,0.2,0.1,0.05,0.025]:
    w = v1 + eps*u; rho = (w@C@w)/(w@w)
    print(f"{eps:>7} {lam[0]-rho:>12.5f} {(lam[0]-rho)/eps**2:>14.4f}")

## 7. The one ceiling you cannot work around

The normalized efficiency $\rho_{\text{norm}}=\lambda_1/\sum_i\lambda_i$ is not an algorithmic limit but an **information limit** — the most variance a single linear unit can capture, as unbeatable as Cauchy–Schwarz itself. Everything in §6 only gets you there *faster*; nothing gets you past it. To exceed it you must add capacity, and the governing theorem is **Eckart–Young**: the best rank-$k$ linear capture is

$$\frac{\sum_{i=1}^k\lambda_i}{\sum_i\lambda_i}\qquad(\text{a }k\text{-output subspace network}),$$

while structure that $\mathbf C$ cannot see at all requires a nonlinearity. That is the honest boundary between making the algorithm faster and changing what is achievable.

In [ ]:
cum = np.cumsum(lam)/lam.sum()
plt.figure(figsize=(6,3.8)); plt.plot(range(1,n+1), cum, "o-")
plt.axhline(1,ls=":",color="gray"); plt.xlabel("rank k"); plt.ylabel("captured variance fraction")
plt.title("Eckart-Young: cumulative eigenvalue capture"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print("single unit (k=1) ceiling:", round(cum[0],4), " | k=3:", round(cum[2],4))

## 8. The bridge: the alignment law as Cauchy–Schwarz in parameter space

Everything above lives in a single neuron's *input* space, where $\cos\theta=\mathbf w^\top\mathbf x/(\|\mathbf w\|\|\mathbf x\|)$ measures how well a weight vector aligns with an input direction. Post-training needs the same geometry one level up, in **parameter** space, where the object being aligned is a **zeroth-order gradient estimate** against the true gradient.

**The estimator.** Freeze the backbone; confine all $P$ trainable degrees of freedom to a LoRA subspace, updating with forward passes only. With probe $\mathbf u\sim\mathcal N(0,I_P)$ and step $\mu$, the two-point antithetic estimator of $\mathbf g=\nabla_\theta L$ is

$$\hat{\mathbf g}=\frac{L(\theta+\mu\mathbf u)-L(\theta-\mu\mathbf u)}{2\mu}\,\mathbf u\ \xrightarrow[\mu\to0]{}\ (\mathbf u^\top\mathbf g)\,\mathbf u,$$

unbiased ($\mathbb E[\hat{\mathbf g}]=\mathbf g$) but pointing in a random direction on any single draw.

**Lemma (moments).** For $\mathbf u\sim\mathcal N(0,I_P)$ and fixed $\mathbf g$: $\ \mathbb E[\hat{\mathbf g}]=\mathbf g$ and $\ \mathbb E[\|\hat{\mathbf g}\|^2]=(P+2)\|\mathbf g\|^2$.

*Derivation.* $\mathbb E[\mathbf u\mathbf u^\top]=I$ gives the mean. Decompose $\mathbf u=u_\parallel\hat{\mathbf g}_0+\mathbf u_\perp$ with $\hat{\mathbf g}_0=\mathbf g/\|\mathbf g\|$, $u_\parallel\sim\mathcal N(0,1)$, $\mathbf u_\perp$ standard on the orthogonal $(P-1)$-space. Then $\mathbf u^\top\mathbf g=\|\mathbf g\|u_\parallel$, $\|\mathbf u\|^2=u_\parallel^2+\|\mathbf u_\perp\|^2$, so $\mathbb E[\|\hat{\mathbf g}\|^2]=\|\mathbf g\|^2\,\mathbb E[u_\parallel^2(u_\parallel^2+\|\mathbf u_\perp\|^2)]=\|\mathbf g\|^2(3+(P-1))=(P+2)\|\mathbf g\|^2.$

**The law.** Averaging $M$ probes, $\hat{\mathbf g}_M=\tfrac1M\sum_k\hat{\mathbf g}_k$, the signal-to-energy alignment is

$$\boxed{\ \cos^2\theta=\frac{\big(\mathbb E[\hat{\mathbf g}_M^\top\mathbf g]\big)^2}{\|\mathbf g\|^2\,\mathbb E[\|\hat{\mathbf g}_M\|^2]}=\frac{M}{M+P+1},\qquad P=r(m+n)\ \text{for a rank-}r\text{ adapter.}\ }$$

Both the $P$ and the $+1$ are produced by the estimator's own directional variance. $\cos\theta$ is literally the **Cauchy–Schwarz cosine** between $\hat{\mathbf g}_M$ and $\mathbf g$; $\cos^2\theta\to1$ is the equality condition (linear dependence), reachable only as $M\to\infty$ or $P\to0$. LoRA makes the second route cheap: collapsing $P=r(m+n)$ raises alignment at fixed probe budget.

In [ ]:
def align_curve(D_probe_dim, probe, Ms, reps=4000, seed=1):
    r2 = np.random.default_rng(seed)
    g = r2.standard_normal(D_probe_dim); gn2 = g@g
    out = {}
    for M in Ms:
        num = den = 0.0
        for _ in range(reps):
            acc = np.zeros(D_probe_dim)
            for _ in range(M): acc += probe(g, r2)
            acc /= M; num += acc@g; den += acc@acc
        out[M] = (num/reps)**2 / (gn2 * den/reps)
    return out, g
P = 200
def weight_probe(g, r2):
    u = r2.standard_normal(len(g)); return (u@g)*u
Ms = [1,4,16,64,256]
cw,_ = align_curve(P, weight_probe, Ms)
print(f"weight-space alignment law  (P={P}):")
print(f"{'M':>5} {'measured':>10} {'M/(M+P+1)':>11}")
for M in Ms: print(f"{M:>5} {cw[M]:>10.4f} {M/(M+P+1):>11.4f}")

## 9. The node-space alignment law

The program's stated update is a **three-factor** rule $\Delta w=\eta\,(\text{pre})(\text{post-noise})(\text{broadcast})$ — which is **node** (activity) perturbation in structure, not weight perturbation. Node perturbation injects noise at the *activations*, and its variance scales with the number of *nodes*, not weights. On a LoRA adapter the trainable nodes are few.

**Setup.** For one adapted matrix $h=W_0\mathbf x+B\,(A\mathbf x)$, with bottleneck $\mathbf z=A\mathbf x\in\mathbb R^r$ and output $\mathbf u=B\mathbf z\in\mathbb R^m$. The node gradients $\mathbf g_z=\partial L/\partial\mathbf z$, $\mathbf g_u=\partial L/\partial\mathbf u=$ (with $\mathbf g_z=B^\top\mathbf g_u$) produce the weight gradients exactly: $\partial_A L=\mathbf g_z\mathbf x^\top$, $\partial_B L=\mathbf g_u\mathbf z^\top$. Define the gradient energies $E_A=\|\partial_A L\|_F^2$, $E_B=\|\partial_B L\|_F^2$ and $\beta=E_B/(E_A+E_B)$.

**Per-site scheme.** Perturb each site with its own noise: $\hat{\mathbf g}_z=(\boldsymbol\xi_z^\top\mathbf g_z)\boldsymbol\xi_z$, $\hat{\mathbf g}_u=(\boldsymbol\xi_u^\top\mathbf g_u)\boldsymbol\xi_u$. The induced estimates $\widehat{\partial_A L}=\hat{\mathbf g}_z\mathbf x^\top$, $\widehat{\partial_B L}=\hat{\mathbf g}_u\mathbf z^\top$ are unbiased.

**Theorem (node-space alignment law).**

$$\boxed{\ \cos^2\theta=\frac{M}{M+\bar d+1},\qquad \bar d=(1-\beta)\,r+\beta\,m,\qquad \beta=\frac{E_B}{E_A+E_B}.\ }$$

*Derivation.* Signal: $\mathbb E[\hat{\mathbf g}^\top\mathbf g]=E_A+E_B=\|\mathbf g\|^2$ (unbiased). Energy: by the Lemma applied per site, $\mathbb E[\|\hat{\mathbf g}\|^2]=(r+2)E_A+(m+2)E_B=(E_A+E_B)[(r+2)(1-\beta)+(m+2)\beta]=\|\mathbf g\|^2(\bar d+2)$. Averaging $M$ probes gives $\mathbb E[\|\hat{\mathbf g}_M\|^2]=\|\mathbf g\|^2(M+\bar d+1)/M$, hence the law.

The effective dimension is a **barycenter** of the two site dimensions $r$ and $m$, weighted by where the gradient energy sits — never exceeding $\max(r,m)$, and for LoRA ($r\ll m,n$) satisfying $\bar d\le m\ll r(m+n)=P$. The estimation-dimension ratio is $P/\bar d\ge r$. Node perturbation is the **faithful** instantiation of the three-factor rule; the weight-perturbation delivery is its higher-variance approximation.

**Honest cost / caveat.** The clean law uses two independent site perturbations per probe (two antithetic pairs). A one-pair variant perturbs only the output and back-projects, $\widehat{\partial_A L}=(B^\top\hat{\mathbf g}_u)\mathbf x^\top$; still unbiased but its $A$-part variance carries an extra $\|\mathbf g_u\|^2\|B\|_F^2\|\mathbf x\|^2$ (Wick term), inflating the effective $A$-dimension up to the stable rank of $B$. Report both.

In [ ]:
# node-space law: build a structured adapter gradient and measure both sites
r_, m_, n_ = 8, 64, 100
rr = np.random.default_rng(2)
B = rr.standard_normal((m_, r_))/np.sqrt(r_)
x = rr.standard_normal(n_); z = rr.standard_normal(r_)     # presynaptic activity & bottleneck
g_u = rr.standard_normal(m_); g_z = B.T @ g_u              # chain rule g_z = B^T g_u
gA = np.outer(g_z, x); gB = np.outer(g_u, z)
g = np.concatenate([gA.ravel(), gB.ravel()]); gn2 = g@g
E_A, E_B = (gA**2).sum(), (gB**2).sum()
beta = E_B/(E_A+E_B); dbar = (1-beta)*r_ + beta*m_; P_ = r_*(m_+n_)
def node_probe():
    xz = rr.standard_normal(r_); ghz = (xz@g_z)*xz
    xu = rr.standard_normal(m_); ghu = (xu@g_u)*xu
    return np.concatenate([np.outer(ghz,x).ravel(), np.outer(ghu,z).ravel()])
def curve_node(Ms, reps=3000):
    o={}
    for M in Ms:
        num=den=0.0
        for _ in range(reps):
            acc=np.zeros_like(g)
            for _ in range(M): acc+=node_probe()
            acc/=M; num+=acc@g; den+=acc@acc
        o[M]=(num/reps)**2/(gn2*den/reps)
    return o
Ms=[1,4,16,64,256]; cn=curve_node(Ms)
print(f"beta={beta:.3f}  P=r(m+n)={P_}  dbar=(1-b)r+b m={dbar:.2f}  ({P_/dbar:.0f}x smaller)")
print(f"{'M':>5} {'node meas':>10} {'M/(M+dbar+1)':>13}")
for M in Ms: print(f"{M:>5} {cn[M]:>10.4f} {M/(M+dbar+1):>13.4f}")

## 10. The Stage-B rank-sweep prediction: estimation cost × approximation floor

The full prediction multiplies an **estimation** cost by an **approximation** floor.

**Estimation side.** From the alignment law, to reach target $\cos^2\theta=a$ at rank $r$,

$$M^\ast(r)=\frac{a}{1-a}\big(P(r)+1\big)=\frac{a}{1-a}\big(r(m+n)+1\big)\quad\text{(weight)},\qquad M^\ast_{\text{node}}(r)=\frac{a}{1-a}\big(\bar d(r)+1\big),$$

linear in $r$ with slope $(m+n)$ for weight vs. $(1-\beta)$ for node.

**Approximation side (Eckart–Young).** Under a local quadratic model with isotropic Hessian in the update geometry, the best rank-$r$ adapter is the truncated SVD of the optimal update $\Delta W^\ast$, and the fraction of the optimal loss reduction it captures is

$$f(r)=\frac{\sum_{i\le r}\sigma_i^2}{\sum_i\sigma_i^2},\qquad \sigma_i=\sigma_i(\Delta W^\ast),\qquad \text{residual}\propto\sum_{i>r}\sigma_i^2.$$

**Lower bound (the guarantee).** For *every* probe budget $M$ and every rank-$r$ adapter, achievable loss reduction is at most $f(r)$ of the optimum — no amount of sampling crosses the Eckart–Young floor. This is estimator-independent (a property of $\Delta W^\ast$), so the efficient rank $r^\ast=\min\{r:f(r)\ge\tau\}$ (the **spectral knee**) is the same for weight, node, and the backprop control.

**Rayleigh flatness sharpens the knee.** Because $\Delta W^\ast$ is a critical point, residual loss is quadratic in the missed singular values $\propto\sum_{i>r}\sigma_i^2$, so missing small tail components costs almost nothing — the knee is abrupt.

**Combined:** the two sides pull opposite ways in $r$ (estimation cost rises linearly; captured signal saturates at the knee), so the efficient operating point is the knee. Node perturbation lowers the estimation cost and flattens the over-provisioning penalty **without moving the knee or the guarantee**.

In [ ]:
# Estimation-side slopes: measure effective dimension D_eff(r) via 1/cos^2 = 1 + (D+1)/M
rr2 = np.random.default_rng(3)
m2, n2 = 64, 100
def Deff_for_rank(r, kind):
    B = rr2.standard_normal((m2, r))/np.sqrt(r); x = rr2.standard_normal(n2); z = rr2.standard_normal(r)
    g_u = rr2.standard_normal(m2); g_z = B.T@g_u
    gA=np.outer(g_z,x); gB=np.outer(g_u,z); g=np.concatenate([gA.ravel(),gB.ravel()]); gn2=g@g
    P=r*(m2+n2)
    def wprobe():
        u=rr2.standard_normal(len(g)); return (u@g)*u
    def nprobe():
        xz=rr2.standard_normal(r); xu=rr2.standard_normal(m2)
        return np.concatenate([np.outer((xz@g_z)*xz,x).ravel(), np.outer((xu@g_u)*xu,z).ravel()])
    probe = wprobe if kind=="weight" else nprobe
    Ms=[8,32,128]; inv=[]
    for M in Ms:
        num=den=0.0
        for _ in range(300):
            acc=np.zeros_like(g)
            for _ in range(M): acc+=probe()
            acc/=M; num+=acc@g; den+=acc@acc
        inv.append(gn2*den/300/(num/300)**2)
    slope=np.polyfit([1/M for M in Ms], inv, 1)[0]
    return slope-1.0, P
ranks=[2,4,8,16,32]
Dw=[Deff_for_rank(r,"weight")[0] for r in ranks]
Dn=[Deff_for_rank(r,"node")[0] for r in ranks]
Pv=[r*(m2+n2) for r in ranks]
plt.figure(figsize=(6,4))
plt.plot(ranks,Dw,"o",color="C3",label="weight D_eff"); plt.plot(ranks,Pv,"-",color="C3",alpha=.5,label="P=r(m+n)")
plt.plot(ranks,Dn,"s",color="C0",label="node D_eff")
plt.yscale("log"); plt.xlabel("LoRA rank r"); plt.ylabel("effective dimension"); plt.legend(); plt.grid(alpha=.3,which="both")
plt.title("Estimation cost vs rank"); plt.tight_layout(); plt.show()
print(f"weight slope ~ {np.polyfit(ranks,Dw,1)[0]:.0f} (predicted m+n={m2+n2}); node slope ~ {np.polyfit(ranks,Dn,1)[0]:.2f} (~1-beta)")

## 11. Variance-reduction levers that lower $M^\ast$

The benchmark currency is $M^\ast$, the probes to reach a target alignment; $M^\ast=\tfrac{a}{1-a}(D_{\text{eff}}+1)$, so any reduction in the *effective* variance slides $M^\ast$ down.

- **Polyak–Ruppert averaging** — run a slightly larger constant step and report $\bar\theta_T=\tfrac1T\sum_{t\le T}\theta_t$; free law-of-large-numbers variance reduction, the most robust lever in the noisy streaming regime.
- **Control variates and probe design** — a baseline correlated with the directional derivative, and orthogonalized (rather than i.i.d. Gaussian) antithetic probes, shrink $\mathbb E[\|\hat{\mathbf g}_M\|^2]$ toward its signal floor; this is the "minimize the Gram-determinant slack" move applied to the estimator.
- **Momentum / Chebyshev, confined to the spectral core** — where the loss is genuinely quadratic with a spectrum (per-mode conditioning, condition number $\kappa$), heavy-ball turns $O(\kappa)$ into $O(\sqrt\kappa)$ steps. This is the one place the $\sqrt{}$-speedup is rigorous.

**Scope (read this before quoting anything).** Oja's *rule* does **not** transfer to the zeroth-order setting — it is a correlational rule using real pre/post activity, whereas post-training uses random parameter perturbations. What transfers is the **alignment / variance / rank geometry**, which is estimator-agnostic. The clean, rigorous statements live in the linear / spectral regime; a deployed nonlinear denoiser inherits them only heuristically.

## 12. What is novel, and how to keep it honest

**The novel object** is not zeroth-order fine-tuning (MeZO, evolution strategies, RL-for-diffusion all exist), not LoRA, not Eckart–Young, and not node perturbation itself (classical: Williams' REINFORCE; Fiete–Seung; Werfel–Xie–Seung, "node perturbation learns $\sim N$ times faster than weight perturbation"). It is the **joint, quantified law**:

- one **energy-weighted node dimension** $\bar d=(1-\beta)r+\beta m$ that sets the estimation cost of an *adapter*, and
- a single predicted $M^\ast(r)$ curve in which the same spectral **knee** of the task's update $\Delta W^\ast$ governs both the estimation cost (alignment law) and the approximation floor (Eckart–Young), with a matching **lower bound** no rank-$r$ adapter can cross at any $M$.

If the zeroth-order sweep and the backprop control share that knee, the rank you need is a property of the **task, not the optimizer** — a clean, quotable result.

**Built-in falsifiability (state before running).**

- *Dimension test:* node alignment that does not fit $M/(M+\bar d+1)$ with the **measured** $\bar d$ falsifies the node law (suspect finite-$\mu$ bias, non-Gaussian noise, or site-coupling leakage).
- *Knee invariance:* the approximation knee $r^\ast$ must coincide across estimators; a moving knee means the estimator is contaminating the approximation floor.
- *Lower-bound trip wire:* any measured loss reduction beyond $f(r)$ means a bug or the local-quadratic / isotropy assumptions fail — investigate, don't celebrate.
- *No hyperparameter fishing:* the predicted curves depend only on the measured singular spectrum $\{\sigma_i\}$, the measured $\beta$, and the chosen target $a$ — there is no post-hoc knob.

## 13. Open proof obligations

The statements above are at *derivation* level; the following are the formalizations that would make them theorems.

1. **Finite-$\mu$, stochastic oracle.** Promote the alignment law (weight and node) to hold for the finite-$\mu$ antithetic estimator with a minibatch loss oracle; track the smoothing bias and the oracle-variance inflation of the effective dimension.
2. **Ratio vs. expectation.** Replace the signal-to-energy $\cos^2\theta$ with $\mathbb E[\cos^2\theta]$ and bound the gap (concentration in the effective dimension).
3. **Anisotropic geometry.** State the Eckart–Young capture and lower bound under the $\mathcal H$-weighted geometry, removing the isotropy assumption; give conditions under which the Frobenius knee and the $\mathcal H$-weighted knee coincide. *(Empirically, these can diverge — the loss knee is the operative one.)*
4. **Single-pair multi-site scheme.** Quantify the cross-site variance term exactly and give conditions under which its effective dimension approaches $r+m$ rather than $\bar d$.
5. **Combined finite-sample bound.** Combine estimation and approximation into $\mathbb E[L]-L^\ast\le(1-f(r))\Gamma+h(a,M,D_{\text{eff}}(r))$ and minimize over $r$ to certify $r^\ast$ as the optimizer of a bound, confirming estimator-independence of the knee.

## 14. How this maps to the experiment notebooks

| Theory (this notebook) | Experiment |
|---|---|
| §8 weight alignment law `M/(M+P+1)` | `Tier0_alignment_laws_walkthrough.ipynb`, `exp0_alignment_laws.py` |
| §9 node-space law `M/(M+d̄+1)` | Tier 0 (node vs weight) and Tier 1 |
| §10 rank sweep + Eckart–Young knee | `Tier1_diffusion_lora_walkthrough.ipynb` (under the denoising loss) |
| §10 estimation slopes / `D_eff(r)` | `Tier2_estimation_cost_vs_rank.ipynb` |

**Notation.** $\mathbf x,\mathbf w$: input, weight; $y=\mathbf w^\top\mathbf x$; $\mathbf C=\mathbb E[\mathbf x\mathbf x^\top]$; $\lambda_i,\mathbf v_i$: eigen-pairs; $\rho$: Rayleigh quotient; $\Delta=(\lambda_1-\lambda_2)/\lambda_1$: relative gap; $P=r(m+n)$: LoRA weight count; $\bar d=(1-\beta)r+\beta m$: node dimension; $\sigma_i$: singular values of $\Delta W^\ast$; $M$: probe budget; $a$: target alignment.

*Every boxed law above is reproduced numerically in the code cells; re-run the notebook to reproduce the tables and figures.*